# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided, step-by-step exploration for loading, inspecting, and processing the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://www.mlcommons.org/croissant/) available at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
md = dataset.metadata
print(f"{md.name}: {md.description}\n\nVersion: {md.version}\nPublished: {md.date_published}\nCite as: {md.cite_as}")

## 2. Data Overview
Review all record sets, fields, and their `@id`s as defined in the Croissant schema.

In Croissant, the main data tables are called **record sets**. Each has an `@id`, a name, and contains **fields** (columns), which also have their own `@id`s.

Let's enumerate available record sets (tables):

In [ ]:
# List all record sets with @id and name

record_sets = md.record_sets
if not record_sets:
    print("No record sets found in this dataset's schema.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}\n    name: {rs.name}\n    description: {rs.description}")
        print("    Fields:")
        for field in rs.fields:
            col_part = f" (column: {field.column.id})" if hasattr(field, 'column') and getattr(field, 'column', None) is not None else ""
            print(f"      field @id: {field.id}, name: {field.name}, type: {field.data_type}{col_part}")
        print("")

**Sample records for each RecordSet**

The following outputs one example record from each available record set using its `@id`. This helps you inspect the structure and content.

In [ ]:
# Show a sample record from each RecordSet

for rs in (record_sets if record_sets else []):
    print(f"\n=== RecordSet {rs.id} ===")
    try:
        samples = list(dataset.records(record_set=rs.id))
        print("Sample record:")
        if samples:
            print(samples[0])
        else:
            print("No records found.")
    except Exception as e:
        print(f"Could not read records: {e}")

## 3. Data Extraction
Load all records of a selected record set into a DataFrame for analysis. We'll choose one of the main record sets (e.g., the primary protein measurement table) by its `@id`.

If the dataset has more than one record set, you can select as needed.

In [ ]:
# Determine which record set to load, by @id (edit this as needed)
if record_sets:
    # Take the first record set as an example
    selected_record_set_id = record_sets[0].id
    print("Using RecordSet @id:", selected_record_set_id)
else:
    raise RuntimeError('No record sets in the dataset schema.')

# Load all data for the selected record set
records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)
print("\nAvailable columns:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records by a numeric field, normalize, group, etc.

We'll demonstrate filtering by one quantitative column. Please update `numeric_field_id` and `group_field_id` as needed, based on the previous output.

In [ ]:
# Pick numeric and group fields for demonstration, using column names from df
display(df.columns)

# Edit these based on df.columns (ideally use the field @id, which often matches the column name)
numeric_field_id = df.columns[1] if len(df.columns) > 1 else df.columns[0]
group_field_id = df.columns[2] if len(df.columns) > 2 else None
print(f"Numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Group field: {group_field_id}")

# Filter and process
if numeric_field_id:
    # Make sure we handle missing/non-numeric values
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].median()  # Use median as a sample filter threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where '{numeric_field_id}' > median ({threshold}):\nRows: {len(filtered_df)}")
    print(filtered_df.head(2))

    # Normalize
    filtered_df[numeric_field_id + "_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head(2))

    # Optional group
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped means by '{group_field_id}':\n", grouped_df.head())

## 5. Visualization
Visualize a distribution or a relationship using the fields loaded into the DataFrame.

We'll plot a histogram of the selected numeric field and, if available, a boxplot grouped by the specified group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field histogram
if numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Grouped boxplot
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, you have:

- Loaded dataset metadata and records using the `mlcroissant` API,
- Enumerated record sets, fields, and inspected a sample record,
- Loaded tabular data into Pandas for flexible analysis,
- Performed simple filtering, normalization, and grouping steps on a numeric field using @id-based references,
- Visualized value distributions and group-level differences.

For more advanced use and schema references, see the [mlcroissant documentation](https://github.com/mlcommons/croissant). You can further tailor this notebook for domain questions relevant to extracellular vesicle protein mass spectrometry analysis.